In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [ ]:
!pip install wandb -qU
!pip install pytorch-msssim

In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import Dataset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import os
import numpy as np
import wandb
from pytorch_msssim import ssim, ms_ssim

In [ ]:
# Weights and Biases used to track progress of run
wandb.login()

In [ ]:
# model architecture for PCCT

class UNet(nn.Module):
    def __init__(self, channels,relu_slope):
        super(UNet, self).__init__()

        self.initial_conv = nn.Sequential(
            nn.Conv2d(6, channels, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope),
        )

        # encoder 1
        self.enc1 = nn.Sequential(
            nn.Conv2d(channels, channels*2, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels*2, channels*2, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # encoder 2
        self.enc2 = nn.Sequential(
            nn.Conv2d(channels*2, channels*4, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels*4, channels*4, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # encoder 3
        self.enc3 = nn.Sequential(
            nn.Conv2d(channels*4, channels*8, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels*8, channels*8, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # encoder 4
        self.enc4 = nn.Sequential(
            nn.Conv2d(channels*8, channels*16, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels*16, channels*16, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(channels*16, channels*32, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels*32, channels*32, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # decoder and upsample 4
        self.up4 = nn.Sequential(
            nn.ConvTranspose2d(channels*32, channels*16, kernel_size=2, stride=2),
            nn.LeakyReLU(relu_slope)
        )
        self.dec4 = nn.Sequential(
            nn.Conv2d(channels*16 + channels*8, channels*16, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels*16, channels*16, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # decoder and upsample 3
        self.up3 = nn.Sequential(
            nn.ConvTranspose2d(channels*16, channels*8, kernel_size=2, stride=2),
            nn.LeakyReLU(relu_slope)
        )
        self.dec3 = nn.Sequential(
            nn.Conv2d(channels*8 + channels*4, channels*8, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels*8, channels*8, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # decoder and upsample 2
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(channels*8, channels*4, kernel_size=2, stride=2),
            nn.LeakyReLU(relu_slope)
        )
        self.dec2 = nn.Sequential(
            nn.Conv2d(channels*4 + channels*2, channels*4, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels*4, channels*4, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # decoder and upsample 1
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(channels*4, channels*2, kernel_size=2, stride=2),
            nn.LeakyReLU(relu_slope)
        )
        self.dec1 = nn.Sequential(
            nn.Conv2d(channels*2 + channels, channels*2, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope),
            nn.Conv2d(channels*2, channels*2, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # final decoder
        self.dec0 = nn.Sequential(
            nn.Conv2d(channels*2, channels, kernel_size=3, padding=1),
            nn.LeakyReLU(relu_slope)
        )

        # final convolution
        self.out_conv = nn.Conv2d(channels, 1, kernel_size=1)

    def forward(self, x):
        e0 = self.initial_conv(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        # bottleneck
        b = self.bottleneck(e4)

        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        d0 = self.dec0(d1)

        # output
        out = self.out_conv(d0)
        return out


In [ ]:
class makeDataset(Dataset):
    def __init__(self,data_dir,transform=None,mode="train"):
        self.transform = transform
        self.data_dir_mode = os.path.join(data_dir,mode)
        self.mode = mode

    def __len__(self):
        if self.mode=="train":
          return 1224 # number of training images
        elif self.mode=="test":
          return 288 # number of test images
        else:
          return 216 # number of validation images

    def __getitem__(self, idx):
        reduced_sinogram_path = os.path.join(self.data_dir_mode,f"{REDUCED_DOSE}", f"{idx}.npy")
        reduced_sinogram = torch.from_numpy(np.load(reduced_sinogram_path)).float()


        full_sinogram_path = os.path.join(self.data_dir_mode,"360", f"{idx}.npy")
        full_sinogram = torch.from_numpy(np.load(full_sinogram_path)).float().unsqueeze(0)

        if self.transform:
            reduced_sinogram = self.transform(reduced_sinogram)
            full_sinogram = self.transform(full_sinogram)
        return reduced_sinogram, full_sinogram

In [ ]:
# hyperparameters
batch_size = 8
lr = 0.0002
epochs = 1000
channels = 16
relu_slope = 0.15

model = UNet(channels, relu_slope) # model
optimizer = optim.AdamW(model.parameters(), lr=lr) # optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# loading the training and validation datasets
REDUCED_DOSE = "45"
train_dataset = makeDataset(data_dir=f"/content/gdrive/MyDrive/Honours_Thesis/1.2_DATA/ChickenDrumstick/360_to_{REDUCED_DOSE}",mode="train")
validate_dataset = makeDataset(data_dir=f"/content/gdrive/MyDrive/Honours_Thesis/1.2_DATA/ChickenDrumstick/360_to_{REDUCED_DOSE}",mode="validate")

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validate_loader = torch.utils.data.DataLoader(validate_dataset, batch_size=batch_size)

In [ ]:
# defining the combined loss function
class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.85):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.l1_loss = nn.L1Loss()

    def forward(self, prediction, target):
        l1 = self.l1_loss(prediction, target)
        ssim_val = ssim(prediction, target, data_range=1.0, size_average=True)
        combined = self.alpha * (1 - ssim_val) + (1 - self.alpha) * l1

        return combined

In [ ]:
loss_type = "Combined_Loss"
loss_function = CombinedLoss(alpha=0.85)
save_modulus = 2 # how often to save the model

In [ ]:
# make the directory to save the results
os.mkdir('/content/gdrive/MyDrive/Honours_Thesis/2_CHICKENDRUMSTICK_MODELS/UNet2.0/45_run3')
save_path = '/content/gdrive/MyDrive/Honours_Thesis/2_CHICKENDRUMSTICK_MODELS/UNet2.0/45_run3'
start_epoch = 0
losses = []
lowest_loss = float('inf')
epoch_lowest_loss = 0


patience = 15 # how many epochs to wait before stopping
min_delta = 1e-6 # the change in loss that is considered an improvement
patience_counter = 0 # initializer the patience counter
early_stop = False
run = wandb.init(
# entity to where the run will be logged
entity="kianagallagher-university-of-victoria",
# wandb project where the run will be logged
project="45_Chicken",
# run name
name="360_to_45 run 3",
# track hyperparameters and run information
config={
    "learning_rate": lr,
    "architecture": "U-Net_512",
    "dataset": "360_to_45",
    "epochs": epochs,
    "alpha": 0.85
},)

# defining additional metrics
run.define_metric("train/*", step_metric="epoch")
run.define_metric("val/*", step_metric="epoch")

In [ ]:
# function to convert the values to an image that can be seen in Weights and Biases interface
def apply_fixed_window(image, vmin=0.0, vmax=0.07):
    windowed = torch.clamp(image, vmin, vmax)
    windowed = (windowed - vmin) / (vmax - vmin)

    return windowed

In [ ]:
model.to(device)
loss_function.to(device)

# making sure the optimizer is on the correct device
for state in optimizer.state.values():
    for k, v in state.items():
        if isinstance(v, torch.Tensor):
            state[k] = v.to(device)

for epoch in range(start_epoch, epochs):
    # TRAINING
    model.train()
    train_loss = 0
    for reduced_sinograms, ground_truth_sinograms in train_loader:
        reduced_sinograms = reduced_sinograms.float().to(device)
        ground_truth_sinograms = ground_truth_sinograms.float().to(device)
        optimizer.zero_grad()
        reconstructed = model(reduced_sinograms)

        # calculating loss and updating model weights and baises
        loss = loss_function(reconstructed, ground_truth_sinograms)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # VALIDATION PHASE
    model.eval()
    val_loss = 0
    with torch.no_grad(): # disables gradient calculation
        for reduced_sinograms, ground_truth_sinograms in validate_loader:
            reduced_sinograms = reduced_sinograms.float().to(device)
            ground_truth_sinograms = ground_truth_sinograms.float().to(device)

            reconstructed = model(reduced_sinograms)
            loss = loss_function(reconstructed, ground_truth_sinograms)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(validate_loader)
    img_to_log = apply_fixed_window(reconstructed[-1].cpu(), vmin=0.0, vmax=0.07)
    gt_to_log = apply_fixed_window(ground_truth_sinograms[-1].cpu(), vmin=0.0, vmax=0.07)

    comparison = torch.cat([img_to_log,gt_to_log], dim=1)

    # logging values to Weights and Biases
    wandb.log({
      "epoch": epoch,
      "train/loss": avg_train_loss,
      "val/loss": avg_val_loss,
      "visuals/reconstructed": wandb.Image(comparison, caption="Left: Output | Right: Ground Truth")
    })

    print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")

    # checking if the validation loss has improved
    if avg_val_loss < (lowest_loss - min_delta):
        lowest_loss = avg_val_loss
        epoch_lowest_loss = epoch + 1
        patience_counter = 0

        best_model_path = f'{save_path}/best_model_{loss_type}_lr:{lr}_batch:{batch_size}_epoch:{epoch+1}.pth'
        torch.save(model.state_dict(), best_model_path)
        print(f"  --> Validation improved. Saving best model.")
    else:
        patience_counter += 1
        print(f"  --> No improvement in Validation for {patience_counter} epochs.")

    # checking if early stopping has been triggered
    if patience_counter >= patience:
        print("*** Early stopping triggered! ***")
        break

    # save the checkpoint data if training needs to be resumed
    checkpoint_data = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'losses': losses,
        'lowest_loss': lowest_loss,
        'epoch_lowest_loss': epoch_lowest_loss,
        'patience': patience,
        'min_delta': min_delta,
        'patience_counter': patience_counter,
        'early_stop': early_stop
    }
    torch.save(checkpoint_data, f'{save_path}/checkpoint.pth')

    # save specific checkpoints
    if ((epoch + 1) % save_modulus == 0):
        milestone_path = f'{save_path}/model_{loss_type}_lr:{lr}_batch:{batch_size}_epoch:{epoch+1}.pth'
        torch.save(model.state_dict(), milestone_path)

run.finish()